In [70]:
!uv pip install numpy pandas matplotlib nltk pythainlp scikit-learn

Using Python 3.11.15 environment at: /home/kami/Projects/NLP/.venv
Checked 6 packages in 11ms


In [71]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /home/kami/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/kami/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/kami/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [72]:
train = pd.read_csv('./financial-news-data/financial-news-train.csv', encoding='utf-8')
test = pd.read_csv('./financial-news-data/financial-news-test.csv', encoding='utf-8')

In [73]:
def clean_text_dataframe(df, drop_duplicates=True):
    # Lowercase
    df['text'] = df['text'].str.lower()

    # Remove quotes
    df['text'] = df['text'].str.replace('"', '', regex=False)

    # Remove links
    df['text'] = df['text'].str.replace(r'https?://\S+', '', regex=True)

    # Remove words that don't start with an alphabetic character
    df['text'] = df['text'].str.replace(r'\b[^a-z\s]\S*', '', regex=True)

    # Remove non-alphabetic characters
    df['text'] = df['text'].str.replace(r'[^a-z\s]', '', regex=True)

    # Normalize whitespace after removals
    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

    if drop_duplicates:
        df.drop_duplicates(subset=['text'], inplace=True)

    # NLTK remove stopwods
    stop_words = set(stopwords.words('english'))
    df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

In [74]:
# Show full text
pd.set_option('display.max_colwidth', None)

clean_text_dataframe(train)
clean_text_dataframe(test, drop_duplicates=False)

train

,text,label
0,thursday biggest analyst calls apple amazon tesla palantir docusign exxon amp,Analyst Update
1,buy las vegas sands travel singapore builds wells fargo says,Analyst Update
2,piper sandler downgrades docusign sell citing elevated risks amid ceo transition,Analyst Update
3,analysts react tesla latest earnings break next electric car maker,Analyst Update
4,netflix peers set return growth analysts say giving one stock upside,Analyst Update
...,...,...
16984,china developers face wall dollar bond payments second half,Treasuries | Corporate Debt
16985,kfw credit line uniper could raised bln eur handelsblatt,Treasuries | Corporate Debt
16987,russian sells bln roubles one repo auction,Treasuries | Corporate Debt
16988,global esg bond issuance posts h dip supranationals cut back,Treasuries | Corporate Debt


In [75]:
# vocabs = set()

# for text in train.text:
#     vocabs.update(text.split())

# print(len(vocabs))
# print(vocabs)

In [76]:
# Create empty feature vectors
# features = pd.DataFrame(np.zeros((len(train), len(vocabs)), dtype=int), columns=sorted(vocabs), index=train.index)

In [77]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=sorted(vocabs))

X_train = vectorizer.fit_transform(train['text'])
X_test = vectorizer.transform(test['text'])

In [78]:
vectorizer.get_feature_names_out()

array(['aa', 'aad', 'aaic', ..., 'zynx', 'zyus', 'zyversa'],
      shape=(21202,), dtype=object)

In [79]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_train = encoder.fit_transform(train['label'])
y_test = encoder.transform(test['label'])

In [80]:
y_test

array([ 0,  0,  0, ..., 19, 19, 19], shape=(4117,))

In [81]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [82]:
pred = model.predict(X_test)

In [83]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

print("Classification Report:")
print(classification_report(
    y_test,
    pred,
    target_names=encoder.classes_,
    digits=3,
    zero_division=0
))
# print("Confusion Matrix:")
# print(confusion_matrix(y_test, pred))
# print("Precision Score:")
# print(precision_score(y_test, pred, average='weighted'))
# print("Recall Score:")
# print(recall_score(y_test, pred, average='weighted'))
# print("F1 Score:")
# print(f1_score(y_test, pred, average='weighted'))

Classification Report:
                             precision    recall  f1-score   support

             Analyst Update      0.957     0.616     0.750        73
     Company | Product News      0.779     0.898     0.834       852
                 Currencies      0.808     0.656     0.724        32
                   Dividend      0.959     0.969     0.964        97
                   Earnings      0.929     0.917     0.923       242
               Energy | Oil      0.823     0.795     0.808       146
        Fed | Central Banks      0.845     0.818     0.831       214
                 Financials      0.856     0.819     0.837       160
     General News | Opinion      0.742     0.726     0.734       336
  Gold | Metals | Materials      0.529     0.692     0.600        13
                        IPO      0.818     0.643     0.720        14
         Legal | Regulation      0.957     0.739     0.834       119
          M&A | Investments      0.851     0.638     0.729       116
          

In [84]:
model_weights = pd.DataFrame(model.coef_, columns=vectorizer.get_feature_names_out(), index=encoder.classes_).T
model_weights

,Analyst Update,Company | Product News,Currencies,Dividend,Earnings,Energy | Oil,Fed | Central Banks,Financials,General News | Opinion,Gold | Metals | Materials,IPO,Legal | Regulation,M&A | Investments,Macro,Markets,Personnel Change,Politics,Stock Commentary,Stock Movement,Treasuries | Corporate Debt
aa,0.576864,-0.100501,-0.014423,-0.007907,-0.126446,-0.029215,-0.031893,-0.006331,-0.098534,-0.010400,-0.004152,-0.022842,-0.027341,-0.195369,-0.092137,-0.023118,-0.029960,0.145083,0.132019,-0.033397
aad,-0.001089,0.102109,-0.000751,-0.000740,-0.001028,-0.001209,-0.001554,-0.001623,-0.043422,-0.000480,-0.000400,-0.005254,-0.002943,-0.001205,-0.001388,-0.026966,-0.002246,-0.004642,-0.003910,-0.001259
aaic,-0.002756,-0.035110,-0.003635,-0.000678,-0.003004,-0.004832,-0.003961,-0.001221,-0.011558,-0.002252,-0.001138,-0.001004,-0.002363,-0.001591,-0.007589,-0.004471,-0.003026,0.140496,-0.047217,-0.003091
aaii,-0.000214,-0.002774,-0.001561,-0.000244,-0.001376,-0.000420,-0.000738,-0.000473,-0.000797,-0.000463,-0.000174,-0.000191,-0.001245,0.111380,-0.000286,-0.000479,-0.000461,-0.098429,-0.000420,-0.000634
aal,-0.155214,0.001165,-0.017413,-0.011824,0.321462,-0.234897,-0.053389,0.075862,-0.258934,-0.012069,-0.007634,-0.038272,-0.061864,-0.246330,-0.070141,-0.087879,-0.047784,0.596776,0.343239,-0.034859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zynerba,-0.000158,-0.008883,-0.000144,-0.000152,-0.000107,-0.000268,-0.000354,-0.000435,-0.002474,-0.000080,-0.000086,-0.003317,-0.013007,-0.000351,-0.000281,-0.000194,-0.000204,-0.000581,0.031655,-0.000579
zynlonta,-0.000074,0.005200,-0.000067,-0.000394,-0.000112,-0.000085,-0.000125,-0.000744,-0.000206,-0.000071,-0.000053,-0.000165,-0.000194,-0.000224,-0.000166,-0.000758,-0.000215,-0.000238,-0.001059,-0.000250
zynx,-0.001879,0.149359,-0.000441,-0.001009,-0.000689,-0.000687,-0.000810,-0.004560,-0.048362,-0.000464,-0.000725,-0.003179,-0.029367,-0.017303,-0.000987,-0.004428,-0.003095,-0.025930,-0.003246,-0.002198
zyus,-0.004767,0.337761,-0.002222,-0.032364,-0.002081,-0.095631,-0.006457,-0.006645,-0.061008,-0.001346,-0.001368,-0.010292,-0.021398,-0.021933,-0.005316,-0.007538,-0.004525,-0.027466,-0.013140,-0.012264


In [85]:
model_weights.to_csv('model_weights.csv', index=True)